# RAGAS 평가 — 전·월세 팩트체커 RAG

검색+생성 파이프라인을 RAGAS로 평가한다. 제안서에서 정한 4개 지표:

| 지표 | 측정 | 필요 필드 |
|---|---|---|
| **Faithfulness** | 답변이 검색 근거에 충실한가 (환각 억제) | question, contexts, answer |
| **ResponseRelevancy** | 답변이 질문 의도에 부합하는가 | question, answer (+임베딩) |
| **LLMContextPrecisionWithoutReference** | 검색 조항이 관련성 높은 순으로 왔는가 | question, contexts, answer |
| **LLMContextRecall** | 필요한 근거를 빠짐없이 검색했는가 | question, contexts, **reference** |

준비물(.env): `DB_URL`, `OPENAI_API_KEY` / 설치: `pip install ragas`

## 0. RAGAS import 오류 우회 (필수 · 맨 먼저 실행)

`ModuleNotFoundError: langchain_community.chat_models.vertexai` 대응. 설치된 ragas 가 로드 시 ChatVertexAI 를 하드 import 하는데 현재 langchain-community 엔 그 경로가 없다. Vertex 를 쓰지 않으므로 스텁으로 통과시킨다. **ragas 를 import 하기 전에** 실행돼야 한다.

In [ ]:
import importlib, sys, types
_leaf = "langchain_community.chat_models.vertexai"
try:
    importlib.import_module(_leaf)
except ModuleNotFoundError:
    import langchain_community.chat_models          # 실제 존재하는 상위 패키지
    _stub = types.ModuleType(_leaf)
    class ChatVertexAI:                             # import 통과용 스텁 (인스턴스화 안 함)
        def __init__(self, *a, **k):
            raise RuntimeError("ChatVertexAI stub — 사용되지 않음")
    _stub.ChatVertexAI = ChatVertexAI
    sys.modules[_leaf] = _stub
    print("shim 적용됨:", _leaf)


## 1. 설치 · 경로 · import

In [ ]:
# !pip install -q ragas

import os, sys

def find_src_core(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        cand = os.path.join(d, "src", "core")
        if os.path.isfile(os.path.join(cand, "vs_method.py")):
            return cand
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None

SRC = find_src_core()
assert SRC, "src/core/vs_method.py 를 찾지 못했습니다."
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
assert os.getenv("DB_URL") and os.getenv("OPENAI_API_KEY"), ".env 에 DB_URL / OPENAI_API_KEY 필요"

import vs_method
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
print("src/core:", SRC)


## 2. 평가할 RAG 함수 (retrieve → generate)

graph 의 검색+생성을 대표하는 최소 파이프라인. contexts 와 answer 가 정렬돼야 RAGAS 가 정확하다.

In [ ]:
conn = vs_method.get_conn()
gen_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
from langchain_openai import ChatOpenAI as _C
fast_llm = _C(model="gpt-4.1-nano", temperature=0)

REWRITE_ATTEMPTS = 2      # 약한 검색이면 쿼리 재작성 후 재검색 (graph 와 동일 루프)
GRADE_WEAK = 0.35         # top-1 이 미만이면 재작성 발동
K = 12

def _search(q, stage):
    return vs_method.search_similar(conn, q, stage=stage, k=K, min_score=0.15)

def rag_answer(question: str, stage: str = None, issues=None, k: int = K):
    """retrieve(누적 병합 + 재작성 루프) → generate. graph 파이프라인을 반영."""
    hits, query = _search(question, stage), question
    for _ in range(REWRITE_ATTEMPTS):
        top = hits[0]["similarity"] if hits else 0.0
        if top >= GRADE_WEAK:
            break
        query = fast_llm.invoke(
            f"원 질문: {query}\n"
            "임대차 법령 검색에 잘 걸리도록, 원 질문과 다른 표현으로 재작성해라. "
            "관련 법령명(주택임대차보호법 등)과 법률 용어(대항력·우선변제권·수선의무·"
            "계약갱신요구권·임차권등기명령 등) 중 해당하는 것을 포함한 한 문장만 출력."
        ).content.strip()
        extra = _search(query, stage)
        seen, merged = set(), []                        # 누적 병합 (교체 X)
        for h in sorted(hits + extra, key=lambda x: -x.get("similarity", 0)):
            key = h.get("content", "")[:120]
            if key not in seen:
                seen.add(key); merged.append(h)
        hits = merged[:K]

    contexts = [h["content"] for h in hits]
    ctx_txt = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts)) or "(근거 없음)"
    answer = gen_llm.invoke(
        "너는 세입자를 돕는 법률 정보 도우미다. 아래 [근거]만 사용해 답하라.\n"
        "규칙: 1) 첫 문장에서 질문에 직접 답하라(질문의 핵심 용어를 그대로 사용해 결론부터). "
        "2) 근거가 부분적이면 그 범위 안에서 최대한 구체적으로 답하고 마지막에 한 문장으로 한계를 밝혀라. "
        "답변 전체를 '알 수 없다'로 끝내지 마라. 3) 근거에 없는 내용은 단정하지 마라.\n\n"
        f"[근거]\n{ctx_txt}\n\n질문: {question}"
    ).content.strip()
    return answer, contexts

# 스모크 테스트
a, c = rag_answer("전입신고는 언제 하나요?", stage="pre")
print("답변:", a[:80], "…\n근거 수:", len(c))


## 3. 평가셋 (eval_set.json 로드)

440문항(pre/post 220/220)을 `eval_set.json` 에서 불러온다. 전량을 RAGAS 로 돌리면 심판 LLM 비용이 크므로 `SAMPLE_N` 으로 **stage 균형 샘플**을 뽑는다(전체는 `SAMPLE_N=None`).

In [ ]:
import json, os, random

# 노트북 위치에서 위로 올라가며 eval_set.json 탐색 (레포 어디에 두든 대응)
def _find_file(name, start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        for root, _dirs, files in os.walk(d):
            if name in files:
                return os.path.join(root, name)
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None

EVAL_PATH = _find_file('eval_set.json') or 'eval_set.json'
raw = json.load(open(EVAL_PATH, encoding='utf-8'))
print('로드:', EVAL_PATH, '·', len(raw), '문항')

SAMPLE_N = 30            # stage 균형 샘플 개수 (None 이면 전체 440 — 비용 주의)
random.seed(42)

def _take(stage, n):
    pool = [r for r in raw if r['stage'] == stage]
    return pool if n is None else random.sample(pool, min(n, len(pool)))

if SAMPLE_N:
    EVAL_SET = _take('pre', SAMPLE_N // 2) + _take('post', SAMPLE_N - SAMPLE_N // 2)
else:
    EVAL_SET = raw

print('평가 대상:', len(EVAL_SET),
      '(pre', sum(r['stage']=='pre' for r in EVAL_SET),
      '/ post', sum(r['stage']=='post' for r in EVAL_SET), ')')
EVAL_SET[0]


## 4. 샘플 생성 (retrieve+generate 실행)

In [ ]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

samples = []
for row in EVAL_SET:
    ans, ctx = rag_answer(row["question"], stage=row.get("stage"))
    samples.append(SingleTurnSample(
        user_input=row["question"],
        retrieved_contexts=ctx,
        response=ans,
        reference=row["reference"],
    ))

dataset = EvaluationDataset(samples=samples)
print("샘플 수:", len(samples))


## 5. 메트릭 · 심판 LLM 설정 후 평가

In [ ]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, LLMContextPrecisionWithoutReference, LLMContextRecall
try:
    from ragas.metrics import ResponseRelevancy            # 최신
except ImportError:
    from ragas.metrics import AnswerRelevancy as ResponseRelevancy  # 구버전 폴백

# 심판(judge) LLM·임베딩 — 평가용 (토큰 비용 발생)
judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0))
judge_emb = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
)

metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithoutReference(),
    LLMContextRecall(),
]

result = evaluate(dataset=dataset, metrics=metrics, llm=judge_llm, embeddings=judge_emb)
result


## 6. 결과 표 · 평균 스코어

In [ ]:
df = result.to_pandas()
df


In [ ]:
# 지표별 평균 (0~1, 높을수록 좋음)
means = df.select_dtypes("number").mean().round(3)
print("=== RAGAS 평균 ===")
for k, v in means.items():
    print(f"  {k:42s} {v}")
means


## 7. 시각화 · 저조 문항 확인

In [ ]:
import matplotlib.pyplot as plt

metric_cols = list(df.select_dtypes("number").columns)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(range(len(metric_cols)), [df[c].mean() for c in metric_cols])
ax.set_xticks(range(len(metric_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in metric_cols], fontsize=8)
ax.set_ylim(0, 1)
ax.axhline(0.7, ls="--", c="crimson", lw=1)   # 참고선 0.7
ax.set_title("RAGAS mean scores")
for i, c in enumerate(metric_cols):
    ax.text(i, df[c].mean() + 0.02, f"{df[c].mean():.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

# 지표별 최저 문항 (개선 대상)
for c in metric_cols:
    i = df[c].idxmin()
    print(f"[{c}] 최저 {df[c][i]:.2f} → Q: {df['user_input'][i][:40]}")


## 해석 가이드

- **Faithfulness** 낮음 → 답변이 근거를 벗어남(환각). 생성 프롬프트를 근거 기반으로 더 조이거나 verify 강화.
- **ResponseRelevancy** 낮음 → 질문 의도와 답이 어긋남. 질의 분석/쿼리 재작성 점검.
- **ContextPrecision** 낮음 → 관련 없는 조항이 상위에 섞임. 메타필터(stage·issue) 강화, k 축소.
- **ContextRecall** 낮음 → 필요한 근거를 못 찾음. 청킹·임베딩·적재량, min_score·k 조정.

참고: text-embedding-3-small + 소규모 코퍼스에선 Context 지표가 0.3~0.6대로 낮게 나오는 게 흔하다.
문항 수를 늘리고(20~50), reference 품질을 높일수록 수치가 안정된다.